> Experiments with [xsa.util](../trapallada/xsa/util.py)  module

In [ ]:
import pathlib
import tempfile
import itertools

import pandas as pd

from astropy.io import fits

from IPython.display import display, Markdown

import trapallada.xsa.access
import trapallada.xsa.util
import trapallada.plot

So that the notebook is aware of modifications in the the `import`ed modules

In [ ]:
%load_ext autoreload
%autoreload 2

## `download_obs` and `extract_filter_tar`

Downloading an observation...

In [ ]:
sample_obs = '0404720301'
with tempfile.TemporaryDirectory(delete=False) as tmp_dir:

    # context's descriptor is turned into a `Pathlib`
    tmp_dir_root = pathlib.Path(tmp_dir)

    tar_file = trapallada.xsa.access.download_obs(sample_obs, output_directory=tmp_dir_root)

...that is afterwards extracted

In [ ]:
sample_obs_dir = trapallada.xsa.util.extract_filter_tar(tar_file)
sample_obs_dir

Notice that only files whose name matches the regular expression below are kept.

In [ ]:
trapallada.xsa.util.re_filename

Then, e.g.,

In [ ]:
trapallada.xsa.util.re_filename.match('P0006010301PNS003SRSPEC0007.FTZ')

is kept but not

In [ ]:
trapallada.xsa.util.re_filename.match('XMM_OM_0760_0001_NIXTMS.FIT')

## `download_extract_filter`

Downloading, extracting, filtering using a single function (data used in Javi's paper)

In [ ]:
obs = '0605950101' # Javi's paper
obs_dir = trapallada.xsa.util.download_extract_filter(obs, output_directory=tmp_dir_root)

The *first few* downloaded files

In [ ]:
list(itertools.islice(obs_dir.glob('*'), 0, 2))

In [ ]:
fits_files = list(obs_dir.glob('*.FTZ'))
fits_files[:3]

A particular file/source is checked

In [ ]:
sample_file = obs_dir / 'P0605950101PNS003SRCTSR8001.FTZ'
assert sample_file.exists()

## `read_fits_file`

The file is read into a `DataFrame` (other files, such as e.g. `P0144090201PNX000SRCTSR0000.FTZ`) might contain extra/different columns)

In [ ]:
df, mask_df = trapallada.xsa.util.read_fits_file(sample_file)
df.head(10)

The *mask* is `True` for those values included in a *good time interval* (GTI)

In [ ]:
mask_df

In [ ]:
df['TIME'].iloc[[0, -1]]

Sampling period can be found as

In [ ]:
df['TIME'][1] - df['TIME'][0]

## `get_fields_from_header`

One can get fields from the *header* of a file by using

In [ ]:
trapallada.xsa.util.get_fields_from_header(sample_file, ['PCOUNT', 'EXPOSURE'])

## `get_position_from_header`

The position being particularly important, there is specific function

In [ ]:
trapallada.xsa.util.get_position_from_header(sample_file)

## `bin_fits_df`

Just the `RATE` column is selected for binning

In [ ]:
rate_df = trapallada.xsa.util.bin_fits_df(df[['TIME', 'RATE']].set_index('TIME'))
print(f'{len(rate_df)=}')
rate_df

It's plotted

In [ ]:
trapallada.plot.df(rate_df, index_type='T')

## `read_fits_files`

All the above FITs files are read into a `DataFrame`

In [ ]:
fits_df, skipped_files_list, header_df, mask_df = trapallada.xsa.util.read_fits_files(fits_files)
fits_df.head()

The *mask* has `True` at every valid measure

In [ ]:
mask_df.sum(axis=0).hist()

In [ ]:
# fits_df2, skipped_files_list, header_df = trapallada.xsa.util.read_fits_files2(fits_files)
# fits_df2.head()

In [ ]:
# v = fits_df2.values.reshape(
#     (fits_df2.shape[0],
#      len(fits_df2.columns.get_level_values('data').unique()),
#      -1), order='F'
# )
# v.shape

In [ ]:
# for i in range(v.shape[2]):
#     assert np.allclose(v[:, 0, i], fits_df.values[:, i], equal_nan=True)
# repacked_fits_df = fits_df2.filter(like='RATE', axis=1)
# repacked_fits_df.columns = repacked_fits_df.columns.droplevel('data')
# assert repacked_fits_df.equals(fits_df)

Some files where a signals couldn't be found were skipped

In [ ]:
skipped_files_list[:5]

In [ ]:
fits_df.index

Information coming from the header of each FIT file (one row per file)

In [ ]:
header_df.head()

Binning is performed

In [ ]:
binned_df = trapallada.xsa.util.bin_fits_df(fits_df, bin_size=63)

A single measurement is extracted

In [ ]:
measurement_df = binned_df.filter(like='RATE', axis=1)
measurement_df.columns = measurement_df.columns.droplevel('data')
measurement_df

In [ ]:
# trapallada.plot.df(binned_df, columns=[pathlib.Path(c).stem for c in fits_df.columns], index_type='T')
trapallada.plot.df(measurement_df, columns=[pathlib.Path(c).stem for c in measurement_df.columns], index_type='T')

In order to get the *most promising* (meaningful) sources from some observation, its associated (*untar*ed) path is explored

In [ ]:
n_sources_per_obs = 3
ranked_files_df = trapallada.xsa.util.rank_obs_files(obs_dir).iloc[:n_sources_per_obs]['full path']
ranked_files_df

# read_tar

High-level function to read a *tar* file into a `DataFrame`

In [ ]:
tar_file_df, tar_file_headers_df, tar_mask_df = trapallada.xsa.util.read_tar(tar_file)
tar_file_df

In [ ]:
tar_file_headers_df

In [ ]:
tar_mask_df.head()

Slicing

In [ ]:
window_size = 200
step = 100
slices, time_slices = trapallada.xsa.util.slice(tar_file_df, window_size=window_size, step=step)
slices.shape, time_slices.shape

First element is the 1st window for the 1st signal, second element is the 1st window for the 2nd signal, and so forth and so on. Hence,

In [ ]:
(time_slices[0] == time_slices[1]).all()

A `DataFrame` with **all** the timestamps

In [ ]:
time_df = pd.DataFrame(time_slices)
time_df.head()

The durations can be easily obtained...

In [ ]:
durations = time_df.iloc[:, -1] - time_df.iloc[:, 0]
durations.head()

...as well as the periods

In [ ]:
periods = time_df.iloc[:, 1] - time_df.iloc[:, 0]
periods.head()

# `tars_to_array`

The *tar* files downloaded above (into a temporary directory)...

In [ ]:
list(tmp_dir_root.glob('*.tar'))

...are turned into an `Array` (and a companion `DataFrame` with metadata)

In [ ]:
signals, metadata_df, headers_df, full_signals, masks  = trapallada.xsa.util.tars_to_array(tmp_dir_root.glob('*.tar'))
signals.shape, metadata_df.shape, headers_df.shape, masks.shape

Metadata for the headers (notice that every FIT file is *sliced* into multiple signals)

In [ ]:
headers_df

Durations:

In [ ]:
(metadata_df['end'] - metadata_df['start']).head()

For one particular relevant observation

In [ ]:
relevant_source = metadata_df['source'] == 'P0605950101PNS003SRCTSR8001.FTZ'
metadata_df[relevant_source]

Plot

In [ ]:
trapallada.plot.signals_array(signals[relevant_source, 0, :][-1:])

The data from the header

In [ ]:
headers_df.xs('P0605950101PNS003SRCTSR8001.FTZ', level='source')

The full signals are available through a dictionary whose *key*s are the different *tar* files

In [ ]:
full_signals.keys()

Each one is associated with 2 `DataFrame`s

In [ ]:
full_signals_df, full_signals_mask_df = full_signals[next(iter(full_signals.keys()))]
display(full_signals_df)
display(Markdown('---'))
display(full_signals_mask_df)

How many wrong values for every signal

In [ ]:
(~full_signals_mask_df).sum(axis=0)

The lenghts of every full signal

In [ ]:
lengths = [len(dfs[0]) for tar_file, dfs in full_signals.items()]
lengths

Signals are slices of those

In [ ]:
signals.shape